In [1]:
!pip install -q ultralytics roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.0/285.0 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 109.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 6.9 MB/s eta 0:00:00


In [2]:
import os
import torch

from ultralytics import YOLO
from roboflow import Roboflow

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
print("PyTorch Version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Warning: GPU not available.")

PyTorch Version: 2.11.0+cu128
GPU Available: True
GPU: Tesla T4


In [4]:
from roboflow import Roboflow

rf = Roboflow(api_key="Hm1MviJSOnb7f43LMRec")

project = rf.workspace("roboflow-universe-projects").project(
    "license-plate-recognition-rxg4e"
)

# Version 11 = 10,125 images
version = project.version(11)

dataset = version.download("yolov11")

DATASET_PATH = dataset.location

print("Dataset downloaded to:")
print(DATASET_PATH)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to License-Plate-Recognition-11 in yolov11:: 100%|██████████| 20262/20262 [00:03<00:00, 6737.35it/s]

Dataset downloaded to:
/content/License-Plate-Recognition-11


In [5]:
import os

print(os.listdir(DATASET_PATH))

['README.roboflow.txt', 'train', 'valid', 'data.yaml', 'README.dataset.txt', 'test']


In [6]:
YAML_PATH = os.path.join(DATASET_PATH, "data.yaml")

print(open(YAML_PATH).read())

train: ../train/images
val: ../valid/images
test: ../test/images

nc: 1
names: ['License_Plate']

roboflow:
  workspace: roboflow-universe-projects
  project: license-plate-recognition-rxg4e
  version: 11
  license: CC BY 4.0
  url: https://universe.roboflow.com/roboflow-universe-projects/license-plate-recognition-rxg4e/dataset/11


In [7]:
for split in ["train", "valid", "test"]:
    image_dir = os.path.join(DATASET_PATH, split, "images")
    print(f"{split}: {len(os.listdir(image_dir))} images")

train: 7057 images
valid: 2048 images
test: 1020 images


In [8]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

In [9]:
results = model.train(
    data=YAML_PATH,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project="license_plate_detection",
    name="yolo11n_baseline"
)

Ultralytics 8.4.121 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/License-Plate-Recognition-11/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n_basel

In [11]:
from ultralytics import YOLO

BEST_MODEL = "/content/runs/detect/license_plate_detection/yolo11n_baseline/weights/best.pt"

best_model = YOLO(BEST_MODEL)

print("Best model loaded successfully!")

Best model loaded successfully!


In [12]:
test_metrics = best_model.val(
    data=YAML_PATH,
    split="test"
)

Ultralytics 8.4.121 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 717.4±368.5 MB/s, size: 40.3 KB)
val: Scanning /content/License-Plate-Recognition-11/test/labels... 1020 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1020/1020 2.1Kit/s 0.5s
val: New cache created: /content/License-Plate-Recognition-11/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 64/64 7.1it/s 9.0s
                   all       1020       1085       0.99      0.946       0.97      0.691
Speed: 1.0ms preprocess, 3.5ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to /content/runs/detect/val


In [22]:
precision = test_metrics.box.mp
recall = test_metrics.box.mr
map50 = test_metrics.box.map50

f1 = 2 * (precision * recall) / (precision + recall)

print("===== FINAL TEST RESULTS =====")
print(f"mAP@50    : {map50 * 100:.2f}%")
print(f"Precision : {precision * 100:.2f}%")
print(f"Recall    : {recall * 100:.2f}%")
print(f"F1 Score  : {f1 * 100:.2f}%")


===== FINAL TEST RESULTS =====
mAP@50    : 96.95%
Precision : 99.03%
Recall    : 94.56%
F1 Score  : 96.75%
